In [69]:
import numpy as np
import csv

In [70]:
# =======================
# 1. leitura do csv
# =======================

def load_data(filename):
    X = []
    y = []

    with open(filename, "r", encoding='utf-8') as f:
        reader = csv.DictReader(f)

        for row in reader:
            # Target
            survived = row["survived"]
            if survived == "":
                continue

            y.append(int(survived))

            # Features selecionadas
            # vamos usar
            # pclass, sex, age, age, sibsp, parch, fare

            pclass = float(row["pclass"]) if row["pclass"] != "" else 0

            # convertendo sexo
            sex = 1 if row["sex"] == "female" else 0

            age = float(row["age"]) if row["age"] != "" else -1

            sibsp = float(row["sibsp"]) if row["sibsp"] != "" else 0
            parch = float(row["parch"]) if row["parch"] != "" else 0
            fare = float(row["fare"]) if row["fare"] != "" else 0

            X.append([pclass, sex, age, sibsp, parch, fare])

    return np.array(X), np.array(y).reshape(-1, 1)


In [71]:
# =======================
# 2. tratamento de dados
# =======================

def fill_missing_age(X):
    ages = X[:, 2]
    mean_age = np.mean(ages[ ages != -1])

    X[:, 2] = np.where(ages == -1, mean_age, ages)
    return X

def normalize(X):
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)
    return (X - mean) / (std + 1e-8)

def add_bias(X):
    ones = np.ones((X.shape[0], 1))
    return np.hstack((ones, X))

In [72]:
# =======================
# 3. funções do modelo
# =======================

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def compute_cost(X, y, beta):
    m = len(y)
    h = sigmoid(X @ beta)
    epsilon = 1e-8

    cost = -(1/m) * np.sum(
        y * np.log(h + epsilon) + (1 - y) * np.log(1 - h + epsilon)
    )

    return cost

def gradient_descent( X, y, beta, lr, epochs):
    m = len(y)

    for i in range(epochs):
        h = sigmoid(X @ beta)
        gradient = (1/m) * (X.T @ (h - y))

        beta = beta - lr * gradient

        if i % 100 == 0:
            print(f"Epoch {i} - Cost: {compute_cost(X, y, beta):.4f}")

    return beta

In [73]:
# =======================
# 4. treinamento
# =======================

def train(X, y, lr, epochs):
    X = fill_missing_age(X)
    X = normalize(X)
    X = add_bias(X)

    beta = np.zeros((X.shape[1], 1))

    beta = gradient_descent(X, y, beta, lr=0.01, epochs=2000)

    return beta, X

In [74]:
# =======================
# 5. predição
# =======================

def predict(X, beta):
    probs = sigmoid(X @ beta)
    return (probs >= 0.5).astype(int)

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

In [75]:
# =======================
# split treino / teste
# =======================

def train_test_split(X, y, test_size=0.2):
    np.random.seed(42)

    indices = np.arange(len(X))
    np.random.shuffle(indices)

    split = int(len(X) * (1 - test_size))

    train_idx = indices[:split]
    test_idx = indices[split:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

In [76]:
# =======================
# 7. execução
# =======================

X, y = load_data("titanic.csv")

X_train, X_test, y_train, y_test = train_test_split(X, y)

beta, X_train_processed = train(X_train, y_train, 0.1, 500)

# processar teste com MESMAS transformações
X_test = fill_missing_age(X_test)
X_test = normalize(X_test)
X_test = add_bias(X_test)

y_pred = predict(X_test, beta)

#X.append([pclass, sex, age, sibsp, parch, fare])
X_pedro = []
X_pedro.append([1, 1, 21, 0, 0, 200])
X_pedro = np.array(X_pedro)
X_pedro = fill_missing_age(X_pedro)
X_pedro = add_bias(X_pedro)

y_pred2 = predict(X_pedro, beta)

print("Pedro = ",y_pred2)

acc = accuracy(y_test, y_pred)

print("\nAcurácia = ",acc)

Epoch 0 - Cost: 0.6918
Epoch 100 - Cost: 0.5970
Epoch 200 - Cost: 0.5460
Epoch 300 - Cost: 0.5160
Epoch 400 - Cost: 0.4970
Epoch 500 - Cost: 0.4844
Epoch 600 - Cost: 0.4757
Epoch 700 - Cost: 0.4694
Epoch 800 - Cost: 0.4648
Epoch 900 - Cost: 0.4614
Epoch 1000 - Cost: 0.4588
Epoch 1100 - Cost: 0.4568
Epoch 1200 - Cost: 0.4552
Epoch 1300 - Cost: 0.4539
Epoch 1400 - Cost: 0.4529
Epoch 1500 - Cost: 0.4521
Epoch 1600 - Cost: 0.4515
Epoch 1700 - Cost: 0.4509
Epoch 1800 - Cost: 0.4505
Epoch 1900 - Cost: 0.4501
Pedro =  [[1]]

Acurácia =  0.7290076335877863
